# Assignment 3 — Track A SLR: analysis notebook
**Paper:** *Beyond the Final Answer: A Systematic Literature Review of Reward Design Against Process Hallucination in RL-Trained Vision-Language Models*
**Author:** Aafreen Zahra Kazmi, Roll No. 26I-7675 (single-member group, permitted by the instructor)
**Course:** Research Methodology (MG 3010), FAST-NUCES

Structure mirrors the paper: data → descriptive synthesis → bibliometrics → sensitivity analysis → framework figure → screening log summary → pilot analysis.
All figures are saved at 400 DPI (PNG) and as vector PDF in `figures/`; the same files are used in the Overleaf project. Random seed = 42.

**Run:** open in Google Colab or Jupyter from this folder, then *Run all*. Inputs are in `data/` (see `data/codebook.md`).

In [1]:
# Setup: install dependencies (Colab already has most of them)
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Create output folders (needed on a fresh setup)
import os
for d in ['figures', 'tables']:
    os.makedirs(d, exist_ok=True)

## 1. Data and descriptive synthesis
`included_studies.csv` holds the 66 included studies with extraction fields and the two-axis framework coding (`row` = verification target, `col` = verification source).

In [3]:
"""Assignment 3 (Track A SLR) analysis: descriptive synthesis, bibliometrics, sensitivity. All figures at 400 DPI.
Inputs: data/included_studies.csv (66 studies), data/screening_log_full.csv (327 screened records)."""
import re, json, itertools, collections
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt, networkx as nx

SEED = 42
OUT = "figures/"
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 8.5, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlesize": 8.5, "axes.labelsize": 8.5, "legend.fontsize": 8, "xtick.labelsize": 8, "ytick.labelsize": 8})
C = {"blue": "#0969da", "orange": "#bc4c00", "green": "#1a7f37", "purple": "#8250df", "gray": "#8c959f", "ink": "#1f2328"}
def save(fig, name):
    fig.savefig(OUT + name + ".png", dpi=400, bbox_inches="tight", facecolor="white")
    fig.savefig(OUT + name + ".pdf", bbox_inches="tight", facecolor="white"); plt.close(fig)

df = pd.read_csv("data/included_studies.csv")
prim = df[df.kind == "Primary"]
res = {}

### B1: publication trend by arXiv submission month

In [4]:
# ---------- B1: publication trend by arXiv submission month ----------
m = df[df.arxiv_month.fillna("").str.startswith("202")].copy()
m = m[m.arxiv_month >= "2025-01"]
months = pd.period_range("2025-01", "2026-10", freq="M").astype(str)
cnt = m.arxiv_month.value_counts().reindex(months, fill_value=0)
fig, ax = plt.subplots(figsize=(3.5, 1.9))
ax.bar(range(len(cnt)), cnt.values, color=C["blue"], width=0.75)
ax.set_xticks(range(0, len(cnt), 3), [f"{x[5:]}/{x[2:4]}" for x in months[::3]])
ax.set_ylabel("Included studies"); ax.set_xlabel("arXiv submission month (MM/YY)")
ax.set_title("Included studies peaked in April–June 2026", loc="left")
save(fig, "fig_trend"); res["trend_2026"] = int(cnt[[x for x in months if x >= "2026-01"]].sum()); res["trend_2025"] = int(cnt[[x for x in months if x < "2026-01"]].sum())

### B2: top venues (named venue, verified or author-stated)

In [5]:
# ---------- B2: top venues (named venue, verified or author-stated) ----------
def venue_name(r):
    n = str(r.venue_note)
    if r.id == "S25": return "IJCV"
    if r.id == "S26": return "IEEE Access"
    if "LNCS" in n: return "Springer LNCS"
    for v in ["CVPR", "ICCV", "ECCV", "ICLR", "NeurIPS", "ICML", "AAAI", "ACL", "EMNLP", "ICONIP", "COLM"]:
        if v.lower() in n.lower(): return v
    return None
df["venue"] = df.apply(venue_name, axis=1)
verified = df.venue_status.isin(["Venue confirmed", "Venue via DBLP/OpenAlex"])
vt = pd.crosstab(df.venue, verified).reindex(columns=[True, False], fill_value=0)
vt = vt.loc[vt.sum(1).sort_values().index]
fig, ax = plt.subplots(figsize=(3.5, 2.6))
ax.barh(vt.index, vt[True], color=C["green"], label="venue verified")
ax.barh(vt.index, vt[False], left=vt[True], color=C["orange"], label="venue stated by authors")
for i_, tot in enumerate(vt.sum(1)): ax.text(tot + 0.1, i_, str(int(tot)), va="center", fontsize=8)
ax.set_xlabel("Included studies"); ax.legend(frameon=False, loc="lower right")
n_pre = int(df.venue.isna().sum())
ax.set_title(f"Named venues of 30 studies ({n_pre} others are preprints)", loc="left")
save(fig, "fig_venues"); res["venues"] = df.venue_status.value_counts().to_dict(); res["venue_names"] = vt.sum(1).to_dict(); res["n_preprint_novenue"] = n_pre

/tmp/ipykernel_418/3022969724.py:13: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  vt = vt.loc[vt.sum(1).sort_values().index]
/tmp/ipykernel_418/3022969724.py:17: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  for i_, tot in enumerate(vt.sum(1)): ax.text(tot + 0.1, i_, str(int(tot)), va="center", fontsize=8)
/tmp/ipykernel_418/3022969724.py:21: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  save(fig, "fig_venues"); res["venues"] = df.venue_status.value_counts().to_dict(); res["venue_names"] = vt.sum(1).to_dict(); res["n_preprint_novenue"] = n_pre


### B6: framework cell by half-year

In [6]:
# ---------- B6: framework cell by half-year ----------
def half(r):
    mth = r.arxiv_month if isinstance(r.arxiv_month, str) and r.arxiv_month else f"{r.year}-06"
    y, mm = mth.split("-"); return f"{y} H{1 if int(mm) <= 6 else 2}"
prim = prim.assign(half=prim.apply(half, axis=1), cellname=prim.row + " | " + prim.col)
cells = ["Answer/confidence | No extra model", "Perception | No extra model", "Perception | Labelled evidence",
         "Perception | External model", "Reasoning text | Labelled evidence", "Reasoning text | External model",
         "Diagnostic study | -", "Test-time PRM | -"]
labels = ["Answer, no extra model", "Perception, no extra model", "Perception, labels", "Perception, external model",
          "Reasoning, labels", "Reasoning, external model", "Diagnostic", "Test-time PRM"]
cols = ["#9ecae1", C["blue"], "#a1d99b", C["green"], "#fdae6b", C["orange"], "#d0d7de", "#6e7781"]
halves = ["2025 H1", "2025 H2", "2026 H1", "2026 H2"]
tab = pd.crosstab(prim.half, prim.cellname).reindex(index=halves, columns=cells, fill_value=0)
fig, ax = plt.subplots(figsize=(3.5, 2.6)); bottom = np.zeros(len(halves))
for c_, l_, col in zip(cells, labels, cols):
    ax.bar(halves, tab[c_].values, bottom=bottom, color=col, label=l_, width=0.6); bottom += tab[c_].values
ax.set_ylabel("Primary studies"); ax.legend(ncol=2, frameon=False, bbox_to_anchor=(0, -0.18), loc="upper left", fontsize=8)
ax.set_title("Reward families by half-year (n = 63 primary studies)", loc="left")
save(fig, "fig_family_time"); res["cells"] = prim.cellname.value_counts().to_dict()

### B3: keyword co-occurrence in titles of the 327 screened records

In [7]:
# ---------- B3: keyword co-occurrence in titles of the 327 screened records ----------
log = pd.read_csv("data/screening_log_full.csv")
titles = log.Title.str.replace(r"(\w)- (\w)", r"\1-\2", regex=True).str.lower()
norm = {r"reinforcement|\brl\b|grpo|policy optimi[sz]ation|rlvr|rft": "RL / policy opt.",
        r"hallucinat": "hallucination", r"reward": "reward", r"faithful|fidelity": "faithfulness",
        r"perception|perceptual|perceive": "perception", r"ground": "grounding", r"reason": "reasoning",
        r"process|step": "process / step", r"caption": "captioning", r"video": "video", r"medical|radiolog|patholog|x-ray|clinic": "medical",
        r"agent": "agent", r"consisten": "consistency", r"attention|look|attend": "visual attention",
        r"calibrat|confiden": "calibration", r"benchmark": "benchmark", r"survey|review": "survey",
        r"preference|dpo": "preference / DPO", r"evidence": "evidence", r"rubric|judge|critic": "judge / rubric",
        r"counterfactual|causal": "counterfactual", r"math": "math", r"spatial": "spatial", r"chart|table": "chart / table"}
terms = [[k for p, k in norm.items() if re.search(p, t)] for t in titles]
freq = collections.Counter(itertools.chain.from_iterable(terms))
co = collections.Counter(itertools.chain.from_iterable(itertools.combinations(sorted(set(x)), 2) for x in terms))
G = nx.Graph()
for k, f in freq.items(): G.add_node(k, f=f)
for (a, b), w in co.items():
    if w >= 4: G.add_edge(a, b, w=w)
G.remove_nodes_from([n for n in list(G) if G.degree(n) == 0])
pos = nx.kamada_kawai_layout(G)
fig, ax = plt.subplots(figsize=(3.5, 3.6)); ax.axis("off")
ws = [G[a][b]["w"] for a, b in G.edges()]
nx.draw_networkx_edges(G, pos, ax=ax, width=[0.3 + w / 12 for w in ws], edge_color=C["gray"], alpha=0.55)
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=[20 + 3.2 * G.nodes[n]["f"] for n in G], node_color=C["blue"], alpha=0.8)
for n, (x, y) in pos.items():
    big = G.nodes[n]["f"] > 90
    ax.text(x + (0.0 if not big else 0.0), y + (-0.17 if big else 0.09 * (1 if y >= -0.2 else -1.25)), n, fontsize=8 if big else 6.5,
            fontweight="bold" if big else "normal", ha="center", va="center", color=C["ink"], bbox=dict(fc="white", ec="none", alpha=0.85, pad=0.4))
ax.set_title("Title-term co-occurrence (327 screened records)", loc="left")
save(fig, "fig_keywords"); res["top_terms"] = freq.most_common(12)

### B5: topic model (LDA, k chosen by perplexity) on screened titles, compared with author decisions

In [8]:
# ---------- B5: topic model (LDA, k chosen by perplexity) on screened titles, compared with author decisions ----------
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import LatentDirichletAllocation
stop = list(ENGLISH_STOP_WORDS) + ["models", "model", "large", "language", "vision", "multimodal", "mllms", "llms", "vlms",
                                   "lvlms", "vision-language", "via", "towards", "learning"]
cv = CountVectorizer(stop_words=stop, min_df=3); Xc = cv.fit_transform(titles)
perp = {}
for k in range(4, 9):
    perp[k] = LatentDirichletAllocation(n_components=k, random_state=SEED, learning_method="batch", max_iter=200).fit(Xc).perplexity(Xc)
kbest = min(perp, key=perp.get)
lda = LatentDirichletAllocation(n_components=kbest, random_state=SEED, learning_method="batch", max_iter=200).fit(Xc)
vocab = np.array(cv.get_feature_names_out()); topic = lda.transform(Xc).argmax(1)
ta = log["Title/ abstract"].str.replace(r"\s+", "", regex=True)
topics = []
for k in range(kbest):
    top = vocab[np.argsort(lda.components_[k])[::-1][:7]]
    idx = topic == k
    topics.append(dict(topic=k + 1, terms=", ".join(top), n=int(idx.sum()), include_rate=float((ta[idx] == "Include").mean())))
pd.DataFrame(topics).to_csv("tables/tab_topics.csv", index=False); res["topics"] = topics; res["lda_perplexity"] = perp; res["lda_k"] = kbest

### B8: evaluation practice

In [9]:
# ---------- B8: evaluation practice ----------
bench = {"MathVista": r"mathvista", "MathVerse": r"mathverse", "MMMU": r"mmmu", "MathVision": r"mathvision|math-vision",
         "HallusionBench": r"hallusionbench", "MMStar": r"mmstar", "WeMath": r"we-?math", "LogicVista": r"logicvista",
         "POPE": r"\bpope\b", "AMBER": r"amber", "ChartQA": r"chartqa", "RealWorldQA": r"realworldqa|mme-real"}
txt = (prim.evaluation.fillna("") + " " + prim.data.fillna("") + " " + prim.findings.fillna("")).str.lower()
bc = {k: int(txt.str.contains(p).sum()) for k, p in bench.items()}
faith = r"hallusionbench|\bpope\b|amber|chair|cceval|mmhal|videohallucer|hallucination benchmark|hallucination and reasoning benchmark|hallucination metric|hallucination test|hallucination-sensitive|hallucination,|faithfulness evaluation|reasoning faithfulness|cot consistency|answer inconsistency|faithful tool|evidence-consistency|evidence consistency|calibration error|\bece\b"
res["faith_direct"] = int(txt.str.contains(faith).sum()); res["faith_ids"] = prim.id[txt.str.contains(faith)].tolist()
res["bench"] = bc
bs = pd.Series(bc).sort_values()
fig, ax = plt.subplots(figsize=(3.5, 2.6))
colors_ = [C["orange"] if k in ("HallusionBench", "POPE", "AMBER") else C["blue"] for k in bs.index]
ax.barh(bs.index, bs.values, color=colors_)
for i, x in enumerate(bs.values): ax.text(x + 0.3, i, str(x), va="center", fontsize=8)
ax.set_xlabel("Primary studies using the benchmark (n = 63)")
ax.set_title("Hallucination benchmarks (orange) are rarely used", loc="left")
save(fig, "fig_benchmarks")

### B7: reported gains

In [10]:
# ---------- B7: reported gains ----------
g = pd.DataFrame([
 ("S05 CFPO", "Perception, no extra model", 3.17, 6.25, "RL baseline"), ("S44 CGC", "Perception, no extra model", 2.90, 2.90, "base model"),
 ("S04 PeRL-VL", "Perception, external model", 5.5, 5.5, "RL baseline"), ("S29 VCap", "Perception, external model", 0.02, 0.02, "RL baseline"),
 ("S07 EASE", "Perception, labels", 2.5, 3.1, "RL baseline"),
 ("S11 DEEPO", "Answer, no extra model", 4.0, 4.0, "RL baseline"), ("S43 Degradation-RL", "Answer, no extra model", 2.3, 2.4, "RL baseline"),
 ("S19 Vision-R1", "Answer, no extra model", 6.0, 6.0, "base model"), ("S21 ThinkLite-VL", "Answer, no extra model", 7.0, 7.0, "base model"),
 ("S13 DRT", "Reasoning, labels", 1.3, 1.3, "base model"),
 ("S42 RLR3", "Reasoning, external model", 4.7, 4.7, "base model"), ("S49 Difference Feedback", "Reasoning, external model", 3.0, 3.0, "RL baseline")],
 columns=["study", "family", "lo", "hi", "vs"])
fam_order = ["Answer, no extra model", "Perception, no extra model", "Perception, labels", "Perception, external model",
             "Reasoning, labels", "Reasoning, external model"]
g["fi"] = g.family.map(fam_order.index); g = g.sort_values(["fi", "study"]).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(3.5, 3.4))
for i, r in g.iterrows():
    col = C["blue"] if r.vs == "RL baseline" else C["orange"]
    ax.plot([r.lo, r.hi], [i, i], color=col, lw=1.4)
    ax.plot((r.lo + r.hi) / 2, i, "o" if r.vs == "RL baseline" else "s", color=col, ms=4, mfc=col if r.vs == "RL baseline" else "white")
ax.set_yticks(range(len(g)), [s for s in g.study], fontsize=8)
ax.invert_yaxis(); ax.set_xlabel("Reported gain (points, as stated by authors)")
ax.plot([], [], "o", color=C["blue"], label="vs. GRPO/DAPO/RLVR baseline"); ax.plot([], [], "s", color=C["orange"], mfc="white", label="vs. untrained base model")
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.35, -0.2), ncol=2, fontsize=8); ax.axvline(0, color=C["gray"], lw=0.6)
short = {"Answer, no extra model": "Answer, no model", "Perception, no extra model": "Perc., no model", "Perception, labels": "Perc., labels",
         "Perception, external model": "Perc., ext. model", "Reasoning, labels": "Reas., labels", "Reasoning, external model": "Reas., ext. model"}
for k, (fam, sub) in enumerate(g.groupby("fi", sort=True)):
    i0, i1 = sub.index.min(), sub.index.max()
    if k % 2 == 0: ax.axhspan(i0 - 0.5, i1 + 0.5, color="#f6f8fa", zorder=0)
    ax.text(7.6, (i0 + i1) / 2, short[sub.family.iloc[0]], fontsize=8, va="center", ha="left", color=C["ink"])
ax.set_xlim(-0.3, 7.5)
ax.set_title("Reported gains overlap across reward families", loc="left")
save(fig, "fig_gains"); g.to_csv("tables/tab_gains.csv", index=False)

### Sensitivity analysis

In [11]:
# ---------- Sensitivity analysis ----------
conf = "S01 S02 S07 S09 S12 S15 S20 S21 S23 S25 S26 S30 S32 S42".split()
cost_ids = ["S13", "S54", "S62", "S66"]
scen = {"S0 Baseline (all 66)": df,
        "S1 Peer-reviewed venue only": df[df.id.isin(conf)],
        "S2 Quality ≥ 75% (drop Medium)": df[df.quality == "High"],
        "S3 Without OpenAlex (drop S45–S66)": df[df.source != "OpenAlex"],
        "S4 Without known items (S21–S23)": df[~df.id.isin(["S21", "S22", "S23"])]}
concl = ["C1 Reasoning × no-extra-model cell empty", "C2 Judge-free methods exist (perception/answer)",
         "C3 Text-checking rewards need labels or a judge", "C4 Cost reported by ≤ 10% of studies",
         "C5 Faithfulness measured directly by a minority", "C6 Evidence that accuracy gains can hide ungrounded reasoning"]
rows, mat, ann = [], [], []
for name, d in scen.items():
    p = d[d.kind == "Primary"]; n = len(p)
    rt = p[p.row == "Reasoning text"]
    t_ = (p.evaluation.fillna("") + " " + p.data.fillna("") + " " + p.findings.fillna("")).str.lower()
    nf = int(t_.str.contains(faith).sum()); nc = int(p.id.isin(cost_ids).sum())
    diag = p.id.isin(["S16", "S41", "S59", "S43"]).sum()
    vals = [1 if (rt.col == "No extra model").sum() == 0 else 0,
            1 if (p.col == "No extra model").sum() > 0 else 0,
            1 if len(rt) > 0 and (rt.col != "No extra model").all() else (0.5 if len(rt) == 0 else 0),
            (1 if nc / max(n, 1) <= 0.10 else 0) if nc > 0 else 0.5,
            1 if nf / max(n, 1) < 0.5 else 0,
            1 if diag >= 2 else (0.5 if diag == 1 else 0)]
    txts = [f"{(rt.col=='No extra model').sum()} studies", f"{(p.col=='No extra model').sum()} studies", f"{len(rt)} text-checking",
            f"{nc}/{n}", f"{nf}/{n}", f"{diag} studies"]
    mat.append(vals); ann.append(txts); rows.append(dict(scenario=name, n_total=len(d), n_primary=n, **{c: t for c, t in zip(concl, txts)}))
pd.DataFrame(rows).to_csv("tables/tab_sensitivity.csv", index=False)
from matplotlib.colors import ListedColormap
fig, ax = plt.subplots(figsize=(7.0, 3.2))
ax.imshow(np.array(mat), cmap=ListedColormap(["#ffebe9", "#fff8c5", "#dafbe1"]), vmin=0, vmax=1, aspect="auto")
for i in range(len(mat)):
    for j in range(len(concl)):
        lab = {1: "holds", 0.5: "weakened", 0: "fails"}[mat[i][j]]
        ax.text(j, i, f"{lab}\n{ann[i][j]}", ha="center", va="center", fontsize=8, color=C["ink"])
wrapped = ["C1 Reasoning \u00d7\nno-model cell\nempty", "C2 Judge-free\nmethods\nexist", "C3 Text checks\nneed labels\nor a judge",
           "C4 Cost\nreported by\n\u2264 10%", "C5 Faithfulness\nmeasured by\na minority", "C6 Gains can\nhide ungrounded\nreasoning"]
ax.set_xticks(range(len(concl)), wrapped, fontsize=8)
ax.set_yticks(range(len(scen)), [f"{k}  (n={len(v)})" for k, v in scen.items()], fontsize=8)
ax.xaxis.tick_top(); ax.tick_params(length=0)
for s in ax.spines.values(): s.set_visible(False)
save(fig, "fig_sensitivity"); res["sens"] = rows

### summary numbers

In [12]:
# ---------- summary numbers ----------
res.update(dict(n=len(df), n_primary=len(prim), code=int((df.q8_code == 1).sum()), code_any=int((df.q8_code > 0).sum()),
                limsec=int((df.q7_limitations == 1).sum()), high=int((df.quality == "High").sum()), medium=int((df.quality == "Medium").sum()),
                years=df.year.value_counts().to_dict(), qwen=int(prim.data.str.contains("Qwen", case=False).sum() + 0)))
json.dump(res, open("tables/analysis_summary.json", "w"), indent=1, default=str)
print(json.dumps({k: v for k, v in res.items() if k not in ("sens", "topics")}, indent=1, default=str))
for t in topics: print(t)
print(pd.DataFrame(rows).to_string())

{
 "trend_2026": 40,
 "trend_2025": 23,
 "venues": {
  "Preprint only": 37,
  "Author-stated venue": 15,
  "Venue confirmed": 9,
  "Venue via DBLP/OpenAlex": 5
 },
 "venue_names": {
  "AAAI": 1,
  "COLM": 1,
  "ICCV": 1,
  "ICONIP": 1,
  "IEEE Access": 1,
  "IJCV": 1,
  "Springer LNCS": 2,
  "ICLR": 3,
  "NeurIPS": 3,
  "ICML": 3,
  "ACL": 4,
  "EMNLP": 4,
  "CVPR": 5
 },
 "n_preprint_novenue": 36,
 "cells": {
  "Perception | No extra model": 19,
  "Answer/confidence | No extra model": 12,
  "Reasoning text | External model": 11,
  "Perception | External model": 8,
  "Reasoning text | Labelled evidence": 5,
  "Perception | Labelled evidence": 4,
  "Diagnostic study | -": 3,
  "Test-time PRM | -": 1
 },
 "top_terms": [
  [
   "reasoning",
   125
  ],
  [
   "RL / policy opt.",
   103
  ],
  [
   "grounding",
   40
  ],
  [
   "reward",
   34
  ],
  [
   "medical",
   34
  ],
  [
   "hallucination",
   28
  ],
  [
   "video",
   25
  ],
  [
   "agent",
   22
  ],
  [
   "process / step",

## 2. Framework figure (Fig. 9 in the paper)

In [13]:
"""Framework figure: verification target x verification source (400 DPI PNG + vector PDF)."""
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
out = "figures/fig_framework"
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 8})
INK, QUIET, EDGE, ACC = "#1f2328", "#57606a", "#8c959f", "#0969da"

W, H = 7.16, 6.85                     # inches (IEEE full text width)
fig = plt.figure(figsize=(W, H))
ax = fig.add_axes([0, 0, 1, 1]); ax.set_xlim(0, W); ax.set_ylim(H, 0); ax.axis("off")

x0, cw, gap = 1.35, 1.85, 0.08        # column start, width, gap
cols = [x0 + i * (cw + gap) for i in range(3)]
rhs = [1.3, 2.0, 1.3]; rows = [0.98, 0.98 + 1.3 + 0.08, 0.98 + 1.3 + 2.0 + 0.16]

ax.text(0.15, 0.28, "No published reward checks the reasoning text without a judge model or labels",
        fontsize=10, fontweight="bold", color=INK, va="center")
ax.text(0.15, 0.50, "Rows: what the reward verifies.  Columns: where the verification signal comes from.  S-numbers = included studies.", fontsize=8, color=QUIET, va="center")

heads = [("No extra model", "rule or internal signal"),
         ("Labelled evidence", "region or rationale labels"),
         ("External model", "judge LLM, rubric or reward model")]
for c, (h, s) in zip(cols, heads):
    ax.text(c + cw / 2, 0.73, h, ha="center", fontsize=9, fontweight="bold", color=INK)
    ax.text(c + cw / 2, 0.88, s, ha="center", fontsize=8, color=QUIET)

rlabels = [("Final answer", "or confidence;\nexact match, IoU"),
           ("Perception", "attention, regions,\nvisual facts"),
           ("Reasoning text", "checked against\nthe image")]
for r, rh, (h, s) in zip(rows, rhs, rlabels):
    ax.text(0.15, r + rh / 2 - 0.12, h, fontsize=9, fontweight="bold", color=INK, va="center")
    ax.text(0.15, r + rh / 2 + 0.12, s, fontsize=8, color=QUIET, va="top")

cells = {
 (0, 0): ["S19 Vision-R1, S20 Visual-RFT", "S21 ThinkLite, S50 DeepEyes", "S11 DEEPO, S43 Degrad.-RL",
          "S10 SAVOR, S32 VL-Calib.", "S30 AnE, S39 Refusal-GRPO", "S52 PROPA, S56 MMBoundary"],
 (0, 1): [], (0, 2): [],
 (1, 0): ["S18 PAPO, S05 CFPO", "S12 VIGIL, S15 VGPO", "S34 GPRO, S35 CPPO",
          "S48 ViCrit, S47 PerceptRL", "S61 PEARL, S06 COPO", "S26 OCS, S40 ConsistRoll",
          "S44 CGC, S14 MIRL", "S31 PFlowNet, S33 SIEVE", "S38 TwSG, S62 Look-Again", "S65 When-to-Look"],
 (1, 1): ["S07 EASE", "S08 Faithful-MR1", "S58 Saliency-R1", "S53 Video-SR"],
 (1, 2): ["S17 Perception-R1, S04 PeRL", "S28 PAGER, S29 VCap", "S36 ARM-Thinker, S37 Long", "S66 RAPID, S51 GWYS"],
 (2, 0): None,
 (2, 1): ["S57 DeFacto", "S63 ViRL", "S64 CAVE", "S13 DRT", "S55 SCOUT"],
 (2, 2): ["S01 PaLMR, S02 Perceval", "S03 CORA, S09 V-Rubrics", "S27 V-tableR1, S42 RLR3",
          "S46 CoS, S49 Diff.-Feedback", "S54 AutoRubric, S60 CodeV", "S45 FaithfulQA-FWS"],
}
for (ri, ci), items in cells.items():
    x, y = cols[ci], rows[ri]; rh = rhs[ri]
    empty_gap = items is None
    ax.add_patch(FancyBboxPatch((x, y), cw, rh, boxstyle="round,pad=0,rounding_size=0.08",
                                fc=((9/255, 105/255, 218/255, 0.10) if empty_gap else "white"),
                                ec=(ACC if empty_gap else EDGE), lw=(1.6 if empty_gap else 0.9)))
    if empty_gap:
        ax.text(x + 0.1, y + 0.22, "EMPTY — gap G1", fontsize=8.5, fontweight="bold", color=ACC)
        ax.text(x + 0.1, y + 0.42, "judge-free, label-free check\nof the reasoning text\nagainst the image",
                fontsize=8, color=INK, va="top")
        ax.text(x + 0.1, y + 0.88, "nearest: S03 CORA (trained\nRM); S61 PEARL\n(perception only)",
                fontsize=8, color=QUIET, va="top")
    elif not items:
        ax.text(x + 0.1, y + 0.22, "none found", fontsize=8, color=QUIET, style="italic")
    else:
        for k, t in enumerate(items):
            ax.text(x + 0.1, y + 0.2 + k * 0.17, t, fontsize=8, color=INK)

ya = rows[-1] + rhs[-1] + 0.25
ax.add_patch(FancyArrowPatch((cols[0], ya), (cols[2] + cw, ya), arrowstyle="-|>", mutation_scale=10,
                             color=EDGE, lw=1))
ax.text(cols[0], ya + 0.2, "Extra model, annotation and compute cost rise from left to right →",
        fontsize=8, color=QUIET)
ax.text(cols[0], ya + 0.42, "Not placed: diagnostic S16, S41, S59; test-time reward model S22; benchmark S23; surveys S24, S25.",
        fontsize=8, color=QUIET)
ax.text(cols[0], ya + 0.64, "Base paper MC-R1 (not in the 66; abstract): final answer × labelled evidence, plus a key-token check.",
        fontsize=8, color=INK)
fig.savefig(out + ".png", dpi=400, facecolor="white")
fig.savefig(out + ".pdf", facecolor="white")
print("saved", out)

saved figures/fig_framework


## 3. Screening log (single reviewer, fully published)
All screening was done by the author. Every title/abstract and full-text decision is in `data/screening_log_full.csv` with its IC/EC code and reason, and the snowballing decisions are in `data/snowballing_log.csv`, so any reader can re-screen the records and compare.

In [14]:
log = pd.read_csv('data/screening_log_full.csv')
log.columns = [' '.join(str(c).replace('- ', '-').replace('/ ', '/').split()) for c in log.columns]
log = log.replace({'Uncertai n': 'Uncertain'})
print(len(log), 'screening decisions')
for c in log.columns:
    if 1 < log[c].nunique() <= 12:
        print('\n' + c); print(log[c].value_counts().to_string())
sb = pd.read_csv('data/snowballing_log.csv')
print('\n', len(sb), 'snowballing records')

327 screening decisions

Year
Year
2026    230
2025     86
2024      7
2023      4

Peer-reviewed verified?
Peer-reviewed verified?
Not checked                     162
No                              108
Yes (IEEE Xplore listing)        36
Yes (ACM DL listing)             14
Yes (publisher listing)           4
Yes (ACL Anthology)               2
Yes (Springer listing, IJCV)      1

Title/abstract
Title/abstract
Exclude      227
Include       81
Uncertain     19

Full text
Full text
Include    66
Exclude     4

Full-text exclusion reason
Full-text exclusion reason
EC3    3
EC4    1

Evidence (page or section)
Evidence (page or section)
Abstract, section headings, benchmark and limitation scan of the arXiv HTML full text                            22
Evaluation datasets paragraph: ChartQA, InfoVQA, DocVQA, SlideVQA, ViDoSeek with 10/50-image retrieval setups     1
arXiv HTML page shows a withdrawal notice for 2510.18619                                                          1
Multimoda

## 4. Pilot validation (content validity)
Anonymised Google Form export in `data/pilot_responses.csv` (19 participants, codes E01–E19; timestamps removed). I-CVI, modified kappa and S-CVI/Ave for the full panel and for participants who had trained or fine-tuned a VLM/LLM; open-answer coding is in `data/pilot_open_coding.csv`.

In [15]:
"""Pilot validation: content validity (I-CVI, S-CVI/Ave, modified kappa*) and open-answer coding.
Input: data/pilot_responses.csv (anonymised Google Form export). Outputs: tables/pilot_cvi.csv,
tables/pilot_profile.csv, figures/fig_pilot.(png|pdf)."""
import math, os, pandas as pd, numpy as np, matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
os.makedirs("tables", exist_ok=True); os.makedirs("figures", exist_ok=True)
plt.rcParams.update({"font.size": 8, "font.family": "DejaVu Sans"})
ITEMS = ["P1", "P2", "P3", "P4", "P5", "P6", "F"]
r = pd.read_csv("data/pilot_responses.csv")
R = pd.DataFrame({i: r[[c for c in r.columns if f"[{i}." in c][0]].astype(str).str.extract(r"^(\d)")[0].astype(int) for i in ITEMS})
R.index = r["code"]
trained = (r["trained_vlm_or_llm"] == "Yes").values

def cvi(M):
    n = len(M); a = (M >= 3).sum(); i = a / n
    pc = np.array([math.comb(n, int(k)) * 0.5 ** n for k in a])
    return a, i, (i - pc) / (1 - pc)

rows = []
for name, M in [("all", R), ("trained", R[trained])]:
    a, i, k = cvi(M)
    for it in ITEMS:
        rows.append(dict(panel=name, n=len(M), item=it, rated_3_or_4=int(a[it]), I_CVI=round(i[it], 2),
                         modified_kappa=round(k[it], 2), median=M[it].median()))
t = pd.DataFrame(rows); t.to_csv("tables/pilot_cvi.csv", index=False)
print(t.pivot(index="item", columns="panel", values=["I_CVI", "modified_kappa", "median"]).to_string())
for name, M in [("all", R), ("trained", R[trained])]:
    print(f"S-CVI/Ave P1-P6 ({name}, n={len(M)}) =", round(cvi(M)[1][ITEMS[:6]].mean(), 2))
prof = pd.DataFrame({"role": r["role"].value_counts(), }).rename_axis("value")
pd.concat([r["role"].value_counts().rename("n").to_frame().assign(field="role"),
           r["ml_experience_years"].value_counts().rename("n").to_frame().assign(field="years"),
           r["trained_vlm_or_llm"].value_counts().rename("n").to_frame().assign(field="trained")]).to_csv("tables/pilot_profile.csv")

# figure: rating heatmap (participants x items) + I-CVI bars
order = np.argsort(-R.sum(axis=1).values, kind="stable")
Rs = R.iloc[order]; tr = trained[order]
fig, (ax, bx) = plt.subplots(1, 2, figsize=(3.5, 3.0), gridspec_kw=dict(width_ratios=[1.25, 1], wspace=0.45))
cmap = ListedColormap(["#d9534f", "#f0ad4e", "#9fd3a8", "#2e8b57"])
ax.imshow(Rs.values, cmap=cmap, vmin=0.5, vmax=4.5, aspect="auto")
ax.set_xticks(range(len(ITEMS))); ax.set_xticklabels(ITEMS)
ax.set_yticks(range(len(Rs))); ax.set_yticklabels([f"{c}{'*' if t_ else ''}" for c, t_ in zip(Rs.index, tr)])
for y in range(len(Rs)):
    for x in range(len(ITEMS)):
        ax.text(x, y, Rs.values[y, x], ha="center", va="center", fontsize=8, color="black")
ax.set_title("Relevance ratings (1-4)", fontsize=8)
ax.tick_params(length=0)
ia = cvi(R)[1]; it_ = cvi(R[trained])[1]; x = np.arange(len(ITEMS))
bx.bar(x - 0.2, ia.values, 0.38, color="#7f8c8d", label=f"all (n={len(R)})")
bx.bar(x + 0.2, it_.values, 0.38, color="#2c6fbb", label=f"trained* (n={trained.sum()})")
bx.axhline(0.78, color="black", lw=0.8, ls="--"); bx.text(-0.45, 0.80, "0.78 cut-off", fontsize=8)
bx.set_xticks(x); bx.set_xticklabels(["1","2","3","4","5","6","F"]); bx.set_xlabel("Item (P1-P6, F)", labelpad=1); bx.set_ylim(0, 1.12); bx.set_ylabel("I-CVI", labelpad=1)
bx.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), frameon=False, fontsize=8, ncol=1)
bx.set_title("Item-level CVI", fontsize=8)
for s_ in ["top", "right"]: bx.spines[s_].set_visible(False)
fig.savefig("figures/fig_pilot.png", dpi=400, bbox_inches="tight"); fig.savefig("figures/fig_pilot.pdf", bbox_inches="tight")
print("saved figures/fig_pilot")

      I_CVI         modified_kappa         median        
panel   all trained            all trained    all trained
item                                                     
F      0.68    0.83           0.67    0.83    3.0     3.0
P1     0.37    0.42           0.30    0.28    2.0     2.0
P2     0.58    0.67           0.51    0.62    3.0     3.0
P3     0.47    0.50           0.36    0.35    2.0     2.5
P4     0.47    0.58           0.36    0.48    2.0     3.0
P5     0.53    0.58           0.43    0.48    3.0     3.0
P6     0.68    0.83           0.67    0.83    3.0     3.0
S-CVI/Ave P1-P6 (all, n=19) = 0.52
S-CVI/Ave P1-P6 (trained, n=12) = 0.6


saved figures/fig_pilot


In [16]:
oc = pd.read_csv('data/pilot_open_coding.csv')
print('Judge-free, label-free reward chosen:', oc.Q1_judge_free_label_free.sum(), 'of', len(oc))
print(oc.Q3_benchmarks_show_faithfulness.value_counts().to_string())

Judge-free, label-free reward chosen: 14 of 19
Q3_benchmarks_show_faithfulness
No                  8
Not fully           5
invalid             2
no answer           2
Mostly no           1
Not sufficiently    1
